# FusionModel 服务器训练 Notebook

这个 notebook 用于在服务器上直接运行 attention / attention_stacking 训练。

- 先修改参数单元
- 再依次执行检查与训练单元
- 不会删除或改动源代码文件


In [1]:
from pathlib import Path
import os
import sys
import shlex
import subprocess

ROOT = Path.cwd().resolve()
if not (ROOT / 'src' / 'run_all_modes.py').exists():
    # 如果 notebook 不在仓库根目录启动，回退到固定路径
    ROOT = Path('/home/shuora/Traffic/FusionModel').resolve()

print('Project root:', ROOT)
print('Python:', sys.executable)


Project root: /home/shuora/Traffic/FusionModel
Python: /usr/bin/python3


In [2]:
# ===== 参数区（按需修改）=====
TASK_NAME = 'mta_multiclass'  # 可选: binary_benign_vs_malicious / ustc_multiclass / mta_multiclass / mfcp_multiclass
MODE = 'attention'  # attention / attention_stacking / all

DATASET_ROOT = str(ROOT / 'ProcessedData')
OUTPUT_DIR = str(ROOT / 'outputs')

EPOCHS = 32
BATCH_SIZE = 32
NUM_WORKERS = 4
PREFETCH_FACTOR = 2
DEVICE = 'auto'  # auto / cpu / cuda:0

# 额外参数示例: ['--preset', 'cic_balanced', '--lr', '3e-4']
EXTRA_ARGS = []

print('TASK_NAME =', TASK_NAME)
print('MODE =', MODE)
print('DATASET_ROOT =', DATASET_ROOT)
print('OUTPUT_DIR =', OUTPUT_DIR)


TASK_NAME = mta_multiclass
MODE = attention
DATASET_ROOT = /home/shuora/Traffic/FusionModel/ProcessedData
OUTPUT_DIR = /home/shuora/Traffic/FusionModel/outputs


In [7]:
# 训练前检查
task_root = Path(DATASET_ROOT) / TASK_NAME
required = [
    task_root / 'image_data' / 'Train',
    task_root / 'image_data' / 'Test',
    task_root / 'pcap_data' / 'Train',
    task_root / 'pcap_data' / 'Test',
]
missing = [str(p) for p in required if not p.exists()]
if missing:
    raise FileNotFoundError('miss path')

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
print('check passed。')


FileNotFoundError: miss path

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 组装命令
cmd = [
    sys.executable,
    str(ROOT / 'src' / 'run_all_modes.py'),
    '--mode', MODE,
    '--dataset_root', DATASET_ROOT,
    '--task_name', TASK_NAME,
    '--epochs', str(EPOCHS),
    '--batch_size', str(BATCH_SIZE),
    '--num_workers', str(NUM_WORKERS),
    '--prefetch_factor', str(PREFETCH_FACTOR),
    '--device', DEVICE,
    '--output_dir', OUTPUT_DIR,
] + list(EXTRA_ARGS)

print('即将执行命令:')
print(' '.join(shlex.quote(x) for x in cmd))


In [ ]:
# 开始训练（会实时打印日志）
result = subprocess.run(cmd, cwd=str(ROOT), check=True)
print('训练完成，returncode =', result.returncode)
